In [12]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr

from utils import load_labels, within_auc
from configs import (TABLES, HEAVY, FIGURES)

ImportError: cannot import name 'knn_scores' from 'utils' (D:\Bioinformatics_projects\proteins_tissues\scripts\utils.py)

In [16]:
SPLIT = "split_pair"
K_GRID = [1, 3, 5, 10, 20, 40, 60, 70, 80, 90, 100]
DIVERGING = LinearSegmentedColormap.from_list("blue_grey_red", ["#2a78d6", "#f0efec", "#e34948"])

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

In [4]:
labels, genes, contexts, parts = load_labels(SPLIT)
parts = {name: labels[labels[SPLIT] == name].reset_index(drop=True) for name in ["train", "val", "test"]}
n_genes, n_ctx = len(genes), len(contexts)

In [5]:
Y = np.full((n_genes, n_ctx), np.nan)
Y[parts["train"]["g"], parts["train"]["c"]] = parts["train"]["y"]
gene_rate = np.nanmean(Y, axis=1)
R = Y - gene_rate[:, None]
varies = np.nanstd(Y, axis=1) > 0
S = pd.DataFrame(R[varies]).corr().to_numpy(copy=True)
np.fill_diagonal(S, np.nan)

In [6]:
S

array([[        nan, -0.08931216, -0.184744  , ..., -0.13385833,
        -0.14857339, -0.07456783],
       [-0.08931216,         nan,  0.08763635, ..., -0.03983682,
         0.08444032, -0.0885968 ],
       [-0.184744  ,  0.08763635,         nan, ...,  0.0993698 ,
         0.27880202, -0.04645646],
       ...,
       [-0.13385833, -0.03983682,  0.0993698 , ...,         nan,
         0.1875456 ,  0.23725781],
       [-0.14857339,  0.08444032,  0.27880202, ...,  0.1875456 ,
                nan,  0.04804208],
       [-0.07456783, -0.0885968 , -0.04645646, ...,  0.23725781,
         0.04804208,         nan]], shape=(75, 75))

In [7]:
nearest = np.nanargmax(S, axis=1)
neighbours = pd.DataFrame({
    "context": contexts,
    "nearest": contexts[nearest],
    "similarity": S[np.arange(n_ctx), nearest],
    "second": contexts[np.argsort(-np.nan_to_num(S, nan=-np.inf), axis=1)[:, 1]],
}).sort_values("similarity", ascending=False)

In [8]:
neighbours

,context,nearest,similarity,second
22,Duodenum | glandular cells,Small intestine | glandular cells,0.705537,Colon | glandular cells
59,Small intestine | glandular cells,Duodenum | glandular cells,0.705537,Rectum | glandular cells
50,Rectum | glandular cells,Colon | glandular cells,0.697424,Duodenum | glandular cells
20,Colon | glandular cells,Rectum | glandular cells,0.697424,Duodenum | glandular cells
39,Lymph node | non-germinal center cells,Tonsil | non-germinal center cells,0.594397,Lymph node | germinal center cells
...,...,...,...,...
34,Liver | cholangiocytes,Liver | hepatocytes,0.148650,Ovary | ovarian stroma cells
46,Parathyroid gland | glandular cells,Pancreas | pancreatic endocrine cells,0.119888,Thyroid gland | glandular cells
45,Pancreas | pancreatic endocrine cells,Parathyroid gland | glandular cells,0.119888,Cerebellum | Purkinje cells
42,Ovary | follicle cells,Testis | cells in seminiferous ducts,0.099013,Soft tissue | chondrocytes


In [13]:
def knn_scores(df, k):
    scores = np.empty(len(df))
    for c in range(n_ctx):
        rows = np.flatnonzero(df["c"].to_numpy() == c)
        if len(rows) == 0:
            continue
        order = np.argsort(-np.nan_to_num(S[c], nan=-np.inf))[: n_ctx - 1]
        yg = Y[df["g"].to_numpy()[rows]][:, order]
        observed = ~np.isnan(yg)
        take = observed & (np.cumsum(observed, axis=1) <= k)
        w = np.clip(S[c, order], 1e-6, None) * take
        num = (np.nan_to_num(yg) * w).sum(axis=1)
        den = w.sum(axis=1)
        fallback = gene_rate[df["g"].to_numpy()[rows]]
        scores[rows] = np.where(den > 0, num / np.where(den > 0, den, 1), fallback)
    return scores

In [17]:
grid = []
for k in K_GRID:
    auc, n = within_auc(parts["val"]["y"], knn_scores(parts["val"], k), parts["val"]["g"])
    grid.append({"k": k, "val_auc_within_gene": auc})
    print(f"k {k:>2}: val {auc:.5f}")
grid = pd.DataFrame(grid)
best_k = int(grid.loc[grid["val_auc_within_gene"].idxmax(), "k"])

k  1: val 0.70634
k  3: val 0.78381
k  5: val 0.80117
k 10: val 0.81652
k 20: val 0.82298
k 40: val 0.82447
k 60: val 0.82463
k 70: val 0.82468
k 80: val 0.82468
k 90: val 0.82468
k 100: val 0.82468


In [19]:
test = parts["test"].copy()
test["knn"] = knn_scores(test, best_k)
pred = pd.read_parquet(HEAVY / "03_mf_pred.parquet").query("part == 'test'")
wide = pred.pivot_table(index="pair_id", columns=["model", "seed"], values="p")
seeds = sorted(pred["seed"].unique())
for model in ["additive", "mf"]:
    for s in seeds:
        test[f"{model}_{s}"] = test["pair_id"].map(wide[(model, s)])

In [20]:
def aucs(df):
    out = {"knn": within_auc(df["y"], df["knn"], df["g"])[0]}
    for model in ["additive", "mf"]:
        out[model] = np.mean([within_auc(df["y"], df[f"{model}_{s}"], df["g"])[0] for s in seeds])
    return out

In [21]:
overall = aucs(test)
share = (overall["knn"] - overall["additive"]) / (overall["mf"] - overall["additive"])

In [22]:
overall

{'knn': 0.821239394365537,
 'additive': np.float64(0.7433293228540029),
 'mf': np.float64(0.8258045698630707)}